# Recommendation System Evaluation

This notebook demonstrates comprehensive evaluation and visualization of Content-Based, Collaborative Filtering, and Hybrid recommendation models.

In [1]:
%cd ..

/home/jovyan/project/film-recommendation


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_processing.data_loader import MovieLensDataLoader
from src.data_processing.splitters import DataSplitter
from src.models.content_based import ContentBasedRecommender, ContentBasedConfig
from src.models.collaborative_filtering import CollaborativeFiltering
from src.models.hybrid import HybridRecommender
from src.models.cascading_hybrid import CascadingHybridRecommender
from src.models.popular_baseline_model import PopularityBaseline
from src.evaluation.evaluator import RecommendationEvaluator
from src.evaluation.visualisation import RecommendationVisualizer
import warnings
warnings.filterwarnings('ignore')
import time

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

In [3]:
## 1. Load and Prepare Data

In [4]:
loader = MovieLensDataLoader("ml-latest")
data_dict = loader.load_data()
await loader.letterboxd_data_async(max_concurrent_requests=100)

movies_df = pd.DataFrame(loader.movie_data)
ratings_df = data_dict['ratings']
genre_features = loader.preprocess_movies()
movies_df = pd.concat([movies_df, genre_features], axis=1)
movies_df = movies_df.dropna().reset_index(drop=True)

print(f"Movies: {len(movies_df)}")
print(f"Ratings: {len(ratings_df)}")
print(f"Users: {ratings_df['userId'].nunique()}")





movies_df.head(10)

INFO:src.data_processing.data_loader:Loading MovieLens dataset...
INFO:src.data_processing.data_loader:Loading existing data from cache: data/processed/movie_metadata.parquet
INFO:src.data_processing.data_loader:Found 1180 missing movies with valid TMDB IDs. Fetching...
INFO:src.data_processing.data_loader:DOWNLOAD SUMMARY: Requested: 1180 | Saved: 0 | Failed: 1180


Movies: 82033
Ratings: 33832162
Users: 330975


,movieId,title,year,cast,main_actor,director,rating,runtime,keywords,vote_count,...,genre_film-noir,genre_horror,genre_imax,genre_musical,genre_mystery,genre_romance,genre_sci-fi,genre_thriller,genre_war,genre_western
0,1.0,Toy Story,1995,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",Tom Hanks,John Lasseter,7.978,81.0,"[rescue, friendship, mission, jealousy, villai...",20023.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,4.0,Waiting to Exhale,1995,"[Whitney Houston, Angela Bassett, Loretta Devi...",Whitney Houston,Forest Whitaker,6.243,127.0,"[based on novel or book, single mother, divorc...",206.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5.0,Father of the Bride Part II,1995,"[Steve Martin, Diane Keaton, Martin Short, Kim...",Steve Martin,Charles Shyer,6.273,106.0,"[daughter, baby, parent child relationship, mi...",812.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,6.0,Heat,1995,"[Al Pacino, Robert De Niro, Val Kilmer, Jon Vo...",Al Pacino,Michael Mann,7.937,170.0,"[robbery, chase, obsession, detective, heist, ...",8403.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,7.0,Sabrina,1995,"[Harrison Ford, Julia Ormond, Greg Kinnear, Na...",Harrison Ford,Sydney Pollack,6.210,127.0,"[chauffeur, sibling relationship, paris, franc...",696.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,8.0,Tom and Huck,1995,"[Jonathan Taylor Thomas, Brad Renfro, Eric Sch...",Jonathan Taylor Thomas,Peter Hewitt,5.300,97.0,"[based on novel or book, mississippi river, ma...",214.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6,9.0,Sudden Death,1995,"[Jean-Claude Van Damme, Powers Boothe, Raymond...",Jean-Claude Van Damme,Peter Hyams,6.034,111.0,"[explosive, hostage, ice hockey, terrorism, vi...",812.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
7,10.0,GoldenEye,1995,"[Pierce Brosnan, Sean Bean, Izabella Scorupco,...",Pierce Brosnan,Martin Campbell,6.903,130.0,"[computer virus, cuba, falsely accused, secret...",4353.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,11.0,The American President,1995,"[Michael Douglas, Annette Bening, Martin Sheen...",Michael Douglas,Rob Reiner,6.539,113.0,"[new love, usa president, the white house, hol...",785.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,12.0,Dracula: Dead and Loving It,1995,"[Leslie Nielsen, Mel Brooks, Amy Yasbeck, Pete...",Leslie Nielsen,Mel Brooks,6.071,88.0,"[vampire, satire, spoof, dracula]",1113.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## 2. Split Data (Temporal Split)

In [5]:

splitter = DataSplitter(ratings_df)
splits = splitter.leave_one_out()
train_df = splits['train']
test_df = splits['test']

print(f"Train set: {len(train_df)} ratings")
print(f"Test set: {len(test_df)} ratings")
print(f"Train users: {train_df['userId'].nunique()}")
print(f"Test users: {test_df['userId'].nunique()}")

Train set: 32332614 ratings
Test set: 1498950 ratings
Train users: 330975
Test users: 322384


In [6]:
# # %% [markdown]
# # ## 7. True Scale Analysis: Sweeping Movie Catalog & Interaction Size
# # This section scales both the item catalog size (movies_df) and the user 
# # interactions (train_df) together to evaluate true matrix growth and calculation overhead.

# # %%
# # Define progressive scaling steps for the item catalog
# catalog_fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
# scale_sweep_results = []

# # Keep configurations identical for consistency
# scale_base_config = ContentBasedConfig(
#     artifacts_dir="data/processed/scale_artifacts",
#     show_progress_bars=False,
#     use_gpu=False,
#     similarity_threshold=0.05,
#     top_k_per_item=40
# )

# print(f"Initial raw structures: {len(movies_df)} movies, {len(train_df)} train interactions.")
# print("Starting progressive catalog size sweep...")

# for frac in catalog_fractions:
#     t_start = time.time()
    
#     # 1. Sample the movie catalog
#     if frac == 1.0:
#         sampled_movies_df = movies_df.copy()
#     else:
#         sampled_movies_df = movies_df.sample(frac=frac, random_state=42).reset_index(drop=True)
        
#     allowed_movie_ids = set(sampled_movies_df['movieId'])
    
#     # 2. Filter interactions to match ONLY the remaining catalog items
#     sampled_train_df = train_df[train_df['movieId'].isin(allowed_movie_ids)].copy()
#     sampled_test_df = test_df[test_df['movieId'].isin(allowed_movie_ids)].copy()
    
#     current_movies = len(sampled_movies_df)
#     current_rows = len(sampled_train_df)
    
#     # Skip evaluation if the sub-slice stripped out too much verification context
#     if current_rows == 0 or sampled_test_df.empty:
#         print(f"Fraction {frac:.1f} skipped (Insufficent inner intersection matching).")
#         continue

#     # 3. Fit the model on the downscaled catalog matrices
#     model = ContentBasedRecommender(config=scale_base_config)
#     model.fit(movies_df=sampled_movies_df, ratings_df=sampled_train_df)
    
#     # 4. Evaluate using the custom suite
#     run_key = f"CBR_Catalog_{current_movies}"
#     evaluator = RecommendationEvaluator(
#         models={run_key: model},
#         train_df=sampled_train_df,
#         test_df=sampled_test_df,
#         relevance_threshold=4.0,
#         user_sample_size=None,
#         item_universe=list(allowed_movie_ids) # restrict universe boundary
#     )
    
#     eval_df = evaluator.evaluate_all_models(k_values=[10], max_recommendations=10)
#     elapsed = time.time() - t_start
    
#     if not eval_df.empty:
#         metrics_row = eval_df.iloc[0].to_dict()
#         metrics_row['catalog_fraction'] = frac
#         metrics_row['movie_count'] = current_movies
#         metrics_row['row_count'] = current_rows
#         metrics_row['fit_duration_sec'] = elapsed
#         metrics_row['matrix_nnz'] = model.similarity_matrix.nnz
#         scale_sweep_results.append(metrics_row)
#         print(f"Catalog: {current_movies} movies | Rows: {current_rows} -> NNZ: {metrics_row['matrix_nnz']} | MAP: {metrics_row['map']:.4f}")

# df_scale_metrics = pd.DataFrame(scale_sweep_results)

# # %% [markdown]
# # ## 8. Visualizing Dynamic Graph Scaling Metrics
# # We plot the true computational scaling curves against the movie catalog size.

# # %%
# fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# # Subplot 1: Accuracy Trends across shrinking catalogs
# axes[0].plot(df_scale_metrics['movie_count'], df_scale_metrics['map'], marker='o', linewidth=2.5, color='teal', label='MAP @ 10')
# axes[0].plot(df_scale_metrics['movie_count'], df_scale_metrics['ndcg'], marker='s', linewidth=2.5, color='darkorange', label='NDCG @ 10')
# axes[0].set_title("Recommendation Accuracy vs. Catalog Movie Count", weight='bold')
# axes[0].set_xlabel("Number of Unique Movies in Dataset")
# axes[0].set_ylabel("Metric Score Balance")
# axes[0].grid(True, linestyle='--', alpha=0.5)
# axes[0].legend()

# # Subplot 2: True Computational Scaling Footprint
# color_time = 'tab:red'
# axes[1].plot(df_scale_metrics['movie_count'], df_scale_metrics['fit_duration_sec'], marker='^', linewidth=2.5, color=color_time, label='Fit Time (s)')
# axes[1].set_title("Computational Footprint Scaling Curves", weight='bold')
# axes[1].set_xlabel("Number of Unique Movies in Dataset")
# axes[1].set_ylabel("Total Build Duration (Seconds)", color=color_time)
# axes[1].tick_params(axis='y', labelcolor=color_time)

# ax2 = axes[1].twinx()
# color_nnz = 'tab:purple'
# ax2.plot(df_scale_metrics['movie_count'], df_scale_metrics['matrix_nnz'], marker='v', linewidth=2.5, color=color_nnz, label='Matrix NNZ')
# ax2.set_ylabel("Sparse Similarity Matrix Size (NNZ Count)", color=color_nnz)
# ax2.tick_params(axis='y', labelcolor=color_nnz)

# axes[1].grid(True, linestyle='--', alpha=0.5)
# plt.tight_layout()
# plt.show()

In [7]:
# %% [markdown]
# ## 9. Initializing the Genetic Algorithm Parameter Optimizer
# This block sets up the DEAP framework parameters and establishes a clean 
# evaluation loop that fits your models on the full dataset boundaries.

# %%
import random
import numpy as np
import pandas as pd
from deap import base, creator, tools, algorithms

# 1. Setup multi-objective maximization (Maximize MAP, Maximize Coverage)
# If 'FitnessMulti' or 'Individual' are already registered, clear them or bypass to avoid duplicate errors
if "FitnessMulti" in creator.__dict__:
    del creator.FitnessMulti
if "Individual" in creator.__dict__:
    del creator.Individual

creator.create("FitnessMulti", base.Fitness, weights=(1.0, 1.0))
creator.create("Individual", list, fitness=creator.FitnessMulti)

toolbox = base.Toolbox()

# 2. Define gene generation ranges (Guarding boundaries tightly)
toolbox.register("attr_keywords", random.uniform, 0.05, 0.50)
toolbox.register("attr_actor", random.uniform, 0.05, 0.40)
toolbox.register("attr_director", random.uniform, 0.05, 0.40)
toolbox.register("attr_genre", random.uniform, 0.05, 0.40)
toolbox.register("attr_threshold", random.uniform, 0.07, 0.18)  # Hard guardrail against the zero-noise exploit
toolbox.register("attr_topk", random.randint, 25, 50)

def create_individual():
    """Generates a valid starting chromosome with normalized content weights."""
    raw_weights = [
        toolbox.attr_keywords(), 
        toolbox.attr_actor(), 
        toolbox.attr_director(), 
        toolbox.attr_genre()
    ]
    total = sum(raw_weights)
    normalized_weights = [w / total for w in raw_weights]
    
    # Pack weights + structural configuration metrics
    return creator.Individual(normalized_weights + [toolbox.attr_threshold(), toolbox.attr_topk()])

toolbox.register("individual", create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# 3. The Recommender Evaluation Wrapper
def evaluate_recommender(individual):

    kw_w = abs(individual[0])
    act_w = abs(individual[1])
    dir_w = abs(individual[2])
    gen_w = abs(individual[3])
    
  
    weight_sum = kw_w + act_w + dir_w + gen_w
    if weight_sum == 0:
        kw_w, act_w, dir_w, gen_w = 0.25, 0.25, 0.25, 0.25
        weight_sum = 1.0
        
  
    kw_w /= weight_sum
    act_w /= weight_sum
    dir_w /= weight_sum
    gen_w /= weight_sum
    
  
    individual[0] = kw_w
    individual[1] = act_w
    individual[2] = dir_w
    individual[3] = gen_w
    
  
    individual[4] = max(0.07, min(0.18, individual[4])) 
    individual[5] = int(max(25, min(50, individual[5]))) 
    
  
    threshold = individual[4]
    topk = int(individual[5])
    
  
    cfg = ContentBasedConfig(
        artifacts_dir="data/processed/ga_artifacts",
        keywords_weight=kw_w,
        main_actor_weight=act_w,
        director_weight=dir_w,
        genre_weight=gen_w,
        similarity_threshold=threshold,
        top_k_per_item=topk,
        show_progress_bars=False,
        use_gpu=False
    )
    
    try:
        model = ContentBasedRecommender(config=cfg)
        model.fit(movies_df=movies_df, ratings_df=train_df)
        
        evaluator = RecommendationEvaluator(
            models={"ga_candidate": model},
            train_df=train_df,
            test_df=test_df,
            relevance_threshold=4.0,
            user_sample_size=200, 
            item_universe=list(movies_df['movieId'].unique())
        )
        
        eval_df = evaluator.evaluate_all_models(k_values=[10], max_recommendations=10)
        
        if eval_df.empty:
            return 0.0, 0.0
            
        return eval_df.iloc[0]['map'], eval_df.iloc[0]['coverage']
        
    except Exception:
        return 0.0, 0.0

# Re-register the fixed function
toolbox.register("evaluate", evaluate_recommender)

toolbox.register("evaluate", evaluate_recommender)

# 4. Define Evolutionary Operators 
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.03, indpb=0.15)
toolbox.register("select", tools.selNSGA2) # Non-dominated Sorting Genetic Algorithm (Pareto Selector)

print("Genetic Algorithm successfully initialized and guarded.")

Genetic Algorithm successfully initialized and guarded.


In [8]:
# # %% Execute the Evolution Loop
# import time

# # 1. Set the size of your ecosystem
# POPULATION_SIZE = 40
# GENERATIONS = 4  # How many breeding cycles to run

# print(f"Evolving {POPULATION_SIZE} configurations over {GENERATIONS} generations...")
# start_time = time.time()

# # 2. Generate the initial random population
# population = toolbox.population(n=POPULATION_SIZE)

# # 3. Run the NSGA-II Evolutionary Algorithm
# # eaMuPlusLambda automatically handles evaluating, crossing over, mutating, and selecting.
# final_pop, logbook = algorithms.eaMuPlusLambda(
#     population=population,
#     toolbox=toolbox,
#     mu=POPULATION_SIZE,      # Number of individuals to maintain in each generation
#     lambda_=POPULATION_SIZE,  # Number of children to produce each generation
#     cxpb=0.6,                 # Crossover probability (60% chance to swap genes)
#     mutpb=0.3,                # Mutation probability (30% chance to tweak a parameter)
#     ngen=GENERATIONS,         # Total loops to run
#     verbose=True              # Prints stats at each generation step
# )

# print(f"\nEvolution completed in {time.time() - start_time:.2f} seconds!")

# # 4. Extract the winning 'Pareto Front' (The configurations that didn't cheat)
# pareto_front = tools.selNSGA2(final_pop, k=5)

# print("\n=== TOP 5 OPTIMAL PARETO CONFIGURATIONS ===")
# for i, ind in enumerate(pareto_front):
#     kw, act, dr, gen, thresh, tk = ind
#     print(f"\nConfig #{i+1} (MAP: {ind.fitness.values[0]:.4f}, Coverage: {ind.fitness.values[1]:.4f})")
#     print(f"  -> Weights   | Keywords: {kw:.3f}, Actor: {act:.3f}, Director: {dr:.3f}, Genre: {gen:.3f}")
#     print(f"  -> Precision | Threshold: {thresh:.4f}, Top-K Neighbors: {int(tk)}")

## 3. Train Models

In [9]:
#tags_df = pd.read_csv('/home/aquathirsty/Desktop/PFE/film-recommendations/data/raw/movielens/ml-latest-small/tags.csv')

# Plug the optimal parameters into your full engine configuration
final_optimized_config = ContentBasedConfig(
    artifacts_dir="data/processed/final_optimized",
    keywords_weight=0.416,
    main_actor_weight=0.47,
    director_weight=0.04,
    genre_weight=0.074,
    cast_weight=0.2139, # Remainder to let weights sum to 1.0
    similarity_threshold=0.1172,
    top_k_per_item=48,
    show_progress_bars=True
)

# Run this through your full evaluator_file pipeline now!

cb_model = ContentBasedRecommender(config=final_optimized_config)
cb_model.fit(movies_df=movies_df,ratings_df=train_df)

print("Content-Based model with tags trained successfully!")

2026-07-01 16:57:41.546 | INFO     | src.models.content_based:fit:608 - Starting model fitting
2026-07-01 16:57:41.551 | INFO     | src.models.content_based:fit:615 - Cleaning text columns
2026-07-01 16:57:41.732 | INFO     | src.models.content_based:fit:619 - Building cast and keyword strings
2026-07-01 16:57:41.733 | INFO     | src.models.content_based:fit:621 - Processing cast entries


Building cast strings:   0%|          | 0/82033 [00:00<?, ?it/s]

Building keyword strings:   0%|          | 0/82033 [00:00<?, ?it/s]

2026-07-01 16:57:41.795 | INFO     | src.models.content_based:fit:628 - Preprocessing numerical and rating features
2026-07-01 16:57:41.797 | INFO     | src.models.content_based:_preprocess_numerical_features:122 - Preprocessing numerical features
2026-07-01 16:57:41.806 | DEBUG    | src.models.content_based:_fit_scaler:108 - Clipping runtime between 4.0 and 172.0
2026-07-01 16:57:41.848 | INFO     | src.models.content_based:_preprocess_actor_director_ratings:139 - Computing actor and director average ratings
2026-07-01 16:57:42.062 | INFO     | src.models.content_based:_preprocess_actor_director_ratings:160 - Actor rating range: 0.0000 - 1.0000
2026-07-01 16:57:42.063 | INFO     | src.models.content_based:fit:632 - Building normalized feature matrix
2026-07-01 16:57:42.064 | INFO     | src.models.content_based:_build_feature_matrix:174 - Building feature matrix
2026-07-01 16:57:42.922 | DEBUG    | src.models.content_based:_build_feature_matrix:197 - TF-IDF shapes: main_actor=(82033, 1

Content-Based model with tags trained successfully!


In [10]:
cf_model = CollaborativeFiltering(k_components=110, random_state=42)
cf_model.fit(df_ratings=train_df)
print("Collaborative Filtering model trained successfully!")

2026-07-01 16:59:45.785 | INFO     | src.models.collaborative_filtering:__init__:94 - CollaborativeFiltering initialised. Explicit-feedback biased ALS model ready for training.
ALS epochs: 100%|██████████| 20/20 [08:45<00:00, 26.27s/it]
--- Step: Training complete ---
  Wall clock time : 529.33 s
  CPU time (user) : 10022.68 s
  Memory RSS      : 8869.2 MB
  Memory VMS      : 20139.3 MB
  CPU usage       : 0.0 %
2026-07-01 17:08:35.119 | INFO     | src.models.collaborative_filtering:fit:175 - Fitting complete | wall=529.33s cpu=10022.76s mem=8869.2MB


Collaborative Filtering model trained successfully!


In [11]:
hybrid_model = HybridRecommender(
    cf_model=cf_model,
    cb_model=cb_model,
    alpha=0.8
)
hybrid_model.fitted(cf_model=cf_model, cb_model=cb_model, movies_df=movies_df, ratings_df=train_df)
print("Hybrid model trained successfully!")

Hybrid model trained successfully!


In [12]:
pop_model = PopularityBaseline()
pop_model.fit(train_df)
print("Popularity model trained successfully!")

Popularity model trained successfully!


In [13]:
cascading_hybrid_model = CascadingHybridRecommender(
    primary_model=cf_model,
    secondary_model=cb_model,
    primary_k=50
)
cascading_hybrid_model.fitted(primary_model=cf_model, secondary_model=cb_model)
print("Cascading hybrid model trained successfully!")

Cascading hybrid model trained successfully!


## 4. Evaluate Models

In [ ]:
models = {
    'Content-Based': cb_model,
    'Collaborative': cf_model,
    'Hybrid': hybrid_model,
    'Cascading Hybrid': cascading_hybrid_model,
    'Popularity': pop_model,
}

evaluator = RecommendationEvaluator(
    models=models,
    train_df=train_df,
    test_df=test_df,
    relevance_threshold=4.0,
    user_sample_size=None,
    random_state=42,
    n_negative_samples=99
)

results_df = evaluator.evaluate_all_models(
    k_values=[1, 2, 5, 8,  10, 15, 20],
    max_recommendations=20
    
)

print("Evaluation completed!")
print(f"Results shape: {results_df.shape}")

2026-07-01 17:10:44.884 | INFO     | src.evaluation.evaluator:evaluate_model:219 - Evaluating 'Content-Based'


Content-Based:   0%|          | 0/158 [00:00<?, ?it/s]

2026-07-01 17:10:44.903 | WARNING  | src.evaluation.evaluator:evaluate_model:292 - Batch error for Content-Based: all arrays must be same length. Falling back to single mode.


In [ ]:
results_df

## 5. Visualizations

In [ ]:
visualizer = RecommendationVisualizer(results_df)

### 5.1 Precision@K Trend

In [ ]:
fig = visualizer.plot_metric_trend('precision', figsize=(10, 6))
plt.show()

### 5.2 Recall@K Trend

In [ ]:
fig = visualizer.plot_metric_trend('recall', figsize=(10, 6))
plt.show()

### 5.3 NDCG@K Trend

In [ ]:
fig = visualizer.plot_metric_trend('ndcg', figsize=(10, 6))
plt.show()

### 5.4 Model Comparison at K=10

In [ ]:
fig = visualizer.plot_model_comparison(k=10, figsize=(12, 7))
plt.show()

### 5.5 Performance Heatmap at K=10

In [ ]:
fig = visualizer.plot_all_metrics_heatmap(k=10, figsize=(10, 8))
plt.show()

### 5.6 Coverage vs Novelty Trade-off

In [ ]:
fig = visualizer.plot_coverage_novelty_tradeoff(k=10, figsize=(10, 7))
plt.show()

### 5.7 Radar Chart Comparison

In [ ]:
fig = visualizer.plot_radar_chart(k=10, figsize=(10, 10))
plt.show()

## 6. Save All Plots

In [ ]:
visualizer.save_all_plots(output_dir="evaluation_plots")
print("All plots saved to evaluation_plots/")

## 7. Summary Statistics

In [ ]:
print("=" * 80)
print("EVALUATION SUMMARY (K=10)")
print("=" * 80)

k10_results = results_df[results_df['k'] == 10]

for model in k10_results['model'].unique():
    model_data = k10_results[k10_results['model'] == model].iloc[0]
    print(f"\n{model.upper()}:")
    print(f"  Precision@10: {model_data['precision']:.4f}")
    print(f"  Recall@10:    {model_data['recall']:.4f}")
    print(f"  NDCG@10:      {model_data['ndcg']:.4f}")
    print(f"  MAP@10:       {model_data['map']:.4f}")
    print(f"  MRR:          {model_data['mrr']:.4f}")
    print(f"  Novelty:      {model_data['novelty']:.4f}")
    print(f"  Coverage:     {model_data['coverage']:.4f}")

print("\n" + "=" * 80)